## 6 Итоговая оценка

**Входы**

- Выбранная модель, порог и test

**Выходы**

- Метрики, графики и разбор ошибок

**Что делаем**

Проверим готовое решение на test. Модель и порог на этом шаге уже не меняем.

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


#### Как читать результаты

| Метрика | Что показывает |
| --- | --- |
| AP | Насколько хорошо модель ставит кредиты с дефолтом выше в списке по риску; больше — лучше |
| ROC-AUC | Как часто кредит с дефолтом получает больший риск, чем кредит без дефолта, при попарном сравнении. 0,5 — случайный порядок, 1 — идеальный |
| Precision | Доля настоящих дефолтов среди отмеченных моделью |
| Recall | Доля найденных дефолтов среди всех настоящих |
| F1 и F2 | Объединяют precision и recall. F1 учитывает их на равных, F2 больше внимания отдаёт recall |
| Accuracy | Доля правильных ответов по всем кредитам |
| Balanced accuracy | Средняя полнота для двух классов: многочисленный класс не перевешивает редкий |
| Brier | Средний квадрат разницы между прогнозом риска и фактическим исходом 0/1; меньше — лучше |

Матрица ошибок показывает числа правильных ответов, пропущенных дефолтов и ложных тревог. ROC-кривая сопоставляет полноту с долей ложных тревог, PR-кривая — точность с полнотой при разных порогах.

**Калибровка** показывает, согласуются ли оценки риска с частотой событий. Например, среди заявок с риском около 20% хотелось бы видеть примерно 20% дефолтов. График ниже только проверяет это: по test ничего не настраиваем.

Чтобы понять разброс метрик, используем **bootstrap**: 300 раз собираем тестовый набор случайным выбором строк с повторами. Число кредитов каждого класса сохраняем; модель и порог не меняем. Показываем границы, между которыми лежат средние 95% полученных значений.

Это ориентир для этой модели и этого test. Он не учитывает новое обучение, возможные повторы клиентов или изменение данных со временем.

In [ ]:
# dev-only: загрузчик Python-кода; в сборке заменяется содержимым файла
exec(compile(__import__('pathlib').Path('src/notebooks/_code/evaluate.py').read_text(encoding='utf-8'), 'src/notebooks/_code/evaluate.py', 'exec'), globals())

#### Что ограничивает выводы

Мы проверили модель на одном случайном разбиении Kaggle-датасета. Не знаем время заявок, период наблюдения дефолта и то, встречается ли один человек несколько раз. Также неизвестно, насколько эти данные похожи на заявки конкретного банка. Редкие грейды представлены небольшим числом записей.

Грейд и ставка должны быть известны до прогноза. Если нужна оценка до их назначения, состав признаков придётся пересмотреть и провести новый эксперимент.

Для практического применения нужны проверки на более поздних и внешних данных, оценка стоимости ошибок и качества для разных групп заёмщиков. Полученные метрики сами по себе не обосновывают автоматический отказ в кредите.